# A risky gamble

Each round, your wealth is multiplied by $2$ or by $\tfrac{1}{4}$, each with probability $\tfrac{1}{2}$. The arithmetic expected return per round is
$$\tfrac{1}{2}(2) + \tfrac{1}{2}(0.25) = 1.125,$$
so a risk-neutral expected-wealth maximizer would happily play this game. Below we simulate $N = 10{,}000$ players, each starting at \$1{,}000 and playing $T = 100$ rounds, and look at where they actually end up.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(544)

In [2]:
initial_investment = 1000
T = 100         # rounds per player
N = 10_000      # players
returns = np.array([2.0, 0.25])

In [3]:
# Vectorized Monte Carlo: each row is one player's T draws.
draws = rng.choice(returns, size=(N, T), p=[0.5, 0.5])
final_values = initial_investment * np.prod(draws, axis=1)

## Expected final wealth: theory vs. simulation

Returns are i.i.d. across rounds, so by independence
$$E[W_T] \;=\; W_0 \cdot \bigl(E[R]\bigr)^T \;=\; 1000 \cdot 1.125^{100}.$$

In [4]:
E_R = 0.5 * 2.0 + 0.5 * 0.25
theoretical_E_WT = initial_investment * E_R ** T
simulated_E_WT  = final_values.mean()

print(f"Theoretical E[W_T]: ${theoretical_E_WT:,.2f}")
print(f"Simulated  E[W_T]:  ${simulated_E_WT:,.2f}")

Theoretical E[W_T]: $130,392,389.71
Simulated  E[W_T]:  $2.18


## Percentiles of the wealth distribution

In [5]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 99.99]
values = np.percentile(final_values, percentiles)

table = pd.DataFrame({
    "Percentile": percentiles,
    "Final wealth": [f"${v:,.2f}" for v in values],
})
print(table.to_string(index=False))

 Percentile Final wealth
       1.00        $0.00
       5.00        $0.00
      10.00        $0.00
      25.00        $0.00
      50.00        $0.00
      75.00        $0.00
      90.00        $0.00
      95.00        $0.00
      99.00        $0.01
      99.90       $31.25
      99.99    $2,001.40


## Top 10 final wealth values among the 10,000 players

In [6]:
top10 = np.sort(final_values)[::-1][:10]
top10_table = pd.DataFrame({
    "Rank": np.arange(1, 11),
    "Final wealth": [f"${v:,.2f}" for v in top10],
})
print(top10_table.to_string(index=False))

 Rank Final wealth
    1   $16,000.00
    2    $2,000.00
    3    $2,000.00
    4      $250.00
    5      $250.00
    6      $250.00
    7      $250.00
    8      $250.00
    9      $250.00
   10       $31.25


**Takeaway.** Arithmetic-mean wealth grows, but *individual* players follow a geometric trajectory, not the average. A risk-neutral expected-wealth criterion endorses this bet; almost everyone who actually plays it goes broke. The next notebook (`Kelly_etc.ipynb`) shows how log utility and the Kelly criterion resolve the puzzle.